In [37]:
import time
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.253")
got.load_models(["face_recognition", "color_recognition"])
got.mechanical_joint_control(0, 70, 70, 500)
print("UGOT ready.")

192.168.1.253:50051
UGOT ready.


In [38]:
got.face_recognition_get_all_names()

['Ryan', 'baddie']

In [3]:
# recognise faces if needed
got.face_recognition_add_name("baddie")

3 seconds until taking a photo
2 seconds until taking a photo
1 seconds until taking a photo
Taking a photo...
Face [baddie] registered successfully



In [43]:
# pick up ball
def centralise():
    while True:
        if got.get_color_total_info()[0] == "Green": # use your own color
            color_x = got.get_color_total_info()[2]
            if color_x > 350: # ball is on the right
                got.mecanum_translate_speed(90, 5)
            elif color_x < 290: # ball is on the left
                got.mecanum_translate_speed(-90, 5)
            else: # ball is in the middle
                got.mecanum_stop()
                break
        else: # if the correct ball is not seen
            got.mecanum_stop()

def approach():
    while True:
        if got.get_color_total_info()[0] == "Green": # use your own color
            width = got.get_color_total_info()[5]
            print(width, end="            \r")
            if width > 195: # ball is too close
                got.mecanum_translate_speed(180, 3) # move back
            elif width < 190: # ball is too far
                got.mecanum_translate_speed(0, 3) # move forward
            else: # ball is in the middle
                got.mecanum_stop()
                break
        else: # if the correct ball is not seen
            got.mecanum_stop()

def pickup():
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(0, -60, 0, 500)
    time.sleep(1)
    got.mechanical_clamp_close()
    time.sleep(1)
    got.mechanical_joint_control(0, 30, 0, 500)

centralise()
approach()
pickup()

In [44]:
# find and centralise face
def find_face():
    running = True
    while running:
        got.mecanum_translate_speed(90, 5) # keep moving right
        faces = got.get_face_recognition_total_info()
        for f in faces:
            if f[0] == "baddie":
                # break 
                got.mecanum_stop()
                running = False

def centralise_face():
    running = True
    while running:
        faces = got.get_face_recognition_total_info()
        for f in faces:
            if f[0] == "baddie":
                x_coord = f[1]
                # exercise: centralise the face, stop moving and break out of the loop once it is in the centre
                if x_coord < 290:
                    got.mecanum_translate_speed(-90, 3)
                elif x_coord > 350:
                    got.mecanum_translate_speed(90, 3)
                else:
                    got.mecanum_stop()
                    running = False

find_face()
centralise_face()

In [45]:
# adjust distance and throw
def approach_face():
    running = True
    while running:
        faces = got.get_face_recognition_total_info()
        for f in faces:
            if f[0] == "baddie":
                width = f[4]
                # print(width, end = "       \r")
                if width > 80:
                    got.mecanum_translate_speed(180, 5)
                elif width < 70:
                    got.mecanum_translate_speed(0, 5)
                else:
                    got.mecanum_stop()
                    running = False

def throw():
    got.mechanical_joint_control(0, 70, 70, 500)
    time.sleep(1)
    got.mechanical_joint_control(0, 0, 0, 100)
    got.mechanical_clamp_release()

approach_face()
throw()